# 🎯 Thực nghiệm HỢP NHẤT — một lần chạy, MỘT baseline cho cả ba thí nghiệm

**Dữ liệu:** D1 = 2018−2026, 46 mã, 4 chỉ số tài chính THÔ (điền trung vị, chặn P/E ≤ 100), +45 ngày. Không ranking.

Mỗi seed, mỗi cấu hình, script `stage2_v5.py` huấn luyện đúng 2 mô hình và cho ra 4 dãy Sharpe **ghép cặp cùng ngày**:

| Dãy | Là gì | Thí nghiệm |
|---|---|---|
| `goc` | Mô hình A (giá+KL), chia đều | **BASELINE duy nhất** |
| `dauvao` | Mô hình thêm 4 chỉ số vào đầu vào, chia đều | TN1 |
| `loc` | Giỏ của A, loại mã ROEA < 10% | TN2 |
| `phanbo` | Giỏ của A, vốn tỉ lệ ROEA gốc | TN3 |

Cấu hình (100,50) + (63,20), seeds 42−45, thước đo Sharpe duy nhất. Tổng ~5h GPU T4.
**File cần trên Drive:** `experiment.py`, `stage2_v5.py`, `prepared_2018_2026.csv`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content/drive/MyDrive/LuanVan2/DELAFO_taichinh
import os, importlib
for f in ["experiment.py", "stage2_v5.py", "prepared_2018_2026.csv"]:
    assert os.path.exists(f), f"❌ Thiếu {f}"
import experiment; importlib.reload(experiment)
assert "C_TC_med" in experiment.SCENARIOS, "❌ experiment.py bản cũ"
assert "dauvao" in open("stage2_v5.py").read(), "❌ stage2_v5.py sai bản"
def done(folder, n_dong):
    f = f"{folder}/all_detail.jsonl"
    return os.path.exists(f) and sum(1 for _ in open(f)) >= n_dong
print("✅ Sẵn sàng")

In [ ]:
%pip install -q scipy scikit-learn pandas
import tensorflow as tf
print("GPU:", tf.config.list_physical_devices('GPU') or "⚠️ KHÔNG có — bật T4 trước")

## Chạy toàn bộ (tự bỏ qua seed đã xong; đứt mạng cứ chạy lại cell)

In [ ]:
for sd in [42, 43, 44, 45]:
    out = f"rg3_all_s{sd}"
    if done(out, 2):
        print(f"⏭️ seed {sd} đã xong"); continue
    !python stage2_v5.py --data prepared_2018_2026.csv --start 2018-04-02 \
        --md "100,50;63,20" --epochs 30 --batch 128 --seed {sd} --out rg3_all_s{sd}

## Bảng kết quả trung tâm — MỘT baseline, ba thí nghiệm

In [ ]:
import json, numpy as np
from scipy import stats

data = {}
for sd in [42, 43, 44, 45]:
    for line in open(f"rg3_all_s{sd}/all_detail.jsonl"):
        r = json.loads(line)
        data.setdefault((r["M"], r["D"]), []).append(r)

for md_, lst in sorted(data.items(), reverse=True):
    goc_all = [np.array(r["goc"]) for r in lst]
    goc_tb = np.mean([g.mean() for g in goc_all])
    print(f"\n========= (M,D)={md_} | BASELINE (không dùng TC): Sharpe = {goc_tb:.4f} =========")
    print(f"   baseline theo seed: {['%.3f' % g.mean() for g in goc_all]}\n")
    for v, ten in [("dauvao", "TN1. TC làm ĐẦU VÀO"), ("loc", "TN2. TC LỌC GIỎ (ROEA<10%)"), ("phanbo", "TN3. TC PHÂN BỔ VỐN")]:
        diffs = [np.array(r[v]) - g for r, g in zip(lst, goc_all)]
        diff = np.concatenate(diffs)
        t, p = stats.ttest_1samp(diff, 0)
        nz = diff[diff != 0]
        w, pw = stats.wilcoxon(nz) if len(nz) > 10 else (0, float("nan"))
        per = [x.mean() for x in diffs]
        npos = sum(1 for x in per if x > 0)
        sh = np.mean([np.mean(r[v]) for r in lst])
        kq = "ĐẠT CHUẨN ✓" if (p < 0.05 and pw < 0.05 and npos in (0, 4)) else "hòa / không đạt"
        print(f"{ten:28s} Sharpe={sh:.4f}  Δ={diff.mean():+.4f}  p_t={p:.5f} p_w={pw:.5f}  {npos}/4  → {kq}")
        print(f"{'':28s} Δ theo seed: {['%+.4f' % x for x in per]}")

print("""
Một baseline duy nhất — mọi Δ trừ thẳng vào cùng một cột gốc, ghép cặp cùng ngày.
Ouput để viết chương kết quả.""")

In [ ]:
!zip -r -q ketqua_honhat.zip rg3_all_s4*
!ls -la ketqua_honhat.zip

## Kiểm chứng bền vững theo kiến trúc — SA-GRU (mô hình tốt nhất của nghiên cứu gốc)
Chạy lại toàn bộ 4 biến thể trên SA-GRU, 4 seeds, chỉ cấu hình chính (100,50).
⚠️ Upload `stage2_v5.py` BẢN MỚI (có tham số --model) lên Drive trước, và cần `experiment.py`
có kiến trúc SA_GRU (bản hiện tại đã có).

In [ ]:
assert "--model" in open("stage2_v5.py").read().replace("'",'"') or "args.model" in open("stage2_v5.py").read() or "a.model" in open("stage2_v5.py").read(), "❌ stage2_v5.py bản cũ"
for sd in [42, 43, 44, 45]:
    out = f"rg3_sagru_s{sd}"
    if done(out, 1):
        print(f"⏭️ seed {sd} đã xong"); continue
    !python stage2_v5.py --data prepared_2018_2026.csv --start 2018-04-02 \
        --md 100,50 --epochs 30 --batch 128 --seed {sd} --model SA_GRU --out rg3_sagru_s{sd}

In [ ]:
# Phán quyết SA-GRU + đối chiếu GRU
import json, numpy as np
from scipy import stats

def bang(prefix, seeds=(42,43,44,45)):
    lst = []
    for sd in seeds:
        for line in open(f"{prefix}_s{sd}/all_detail.jsonl"):
            r = json.loads(line)
            if (r["M"], r["D"]) == (100, 50):
                lst.append(r)
    goc_all = [np.array(r["goc"]) for r in lst]
    print(f"  baseline = {np.mean([g.mean() for g in goc_all]):.4f}  theo seed {['%.3f'%g.mean() for g in goc_all]}")
    for v, ten in [("dauvao","TN1 đầu vào"), ("loc","TN2 lọc"), ("phanbo","TN3 phân bổ")]:
        diffs = [np.array(r[v]) - g for r, g in zip(lst, goc_all)]
        diff = np.concatenate(diffs)
        t, p = stats.ttest_1samp(diff, 0)
        nz = diff[diff != 0]
        w, pw = stats.wilcoxon(nz)
        per = [x.mean() for x in diffs]
        npos = sum(1 for x in per if x > 0)
        print(f"  {ten:12s} Δ={diff.mean():+.4f}  p_t={p:.5f} p_w={pw:.5f}  {npos}/{len(per)}  {['%+.3f'%x for x in per]}")

print("===== SA-GRU (100,50) =====")
bang("rg3_sagru")
print("\n===== GRU (100,50) — chính thức, để đối chiếu =====")
bang("rg3_all", seeds=(42,43,44,45,46,47,48,49))

print("""
Đọc: điều cần so là CHIỀU và phán quyết từng thí nghiệm, không phải con số tuyệt đối.
Nếu SA-GRU cho cùng bức tranh → thêm phụ lục 'kết luận không phụ thuộc kiến trúc'.""")

In [ ]:
!zip -r -q ketqua_sagru.zip rg3_sagru_s4*
!ls -la ketqua_sagru.zip

## Kiểm tra độ vững theo số epoch — 300 epoch như nghiên cứu nền
**ĐĂNG KÝ TRƯỚC (đọc trước khi chạy):** phép chạy này CHỈ trả lời một câu hỏi — *chiều kết luận của TN1/TN2/TN3 có giữ nguyên khi tăng epoch từ 30 lên 300 không*. Dù kết quả ra sao cũng KHÔNG thay số liệu chính thức của luận văn (30 epoch); chỉ dùng làm phép kiểm tra độ vững (một câu ở mục 4.6.3) hoặc trả lời miệng trước hội đồng.
Chỉ chạy cấu hình chính (100, 50). Ước tính ~25–30 phút/seed trên T4 → cả 8 seed khoảng 3–4 giờ; ô tự bỏ qua seed đã xong, đứt mạng cứ chạy lại.

In [ ]:
for sd in [42, 43, 44, 45, 46, 47, 48, 49]:
    out = f"rg4_e300_s{sd}"
    if done(out, 1):
        print(f"⏭️ seed {sd} đã xong"); continue
    !python stage2_v5.py --data prepared_2018_2026.csv --start 2018-04-02 \
        --md "100,50" --epochs 300 --batch 128 --seed {sd} --out rg4_e300_s{sd}

In [ ]:
# So sánh nhanh 30 vs 300 epoch — chiều kết luận có giữ nguyên?
import json, glob, numpy as np
from scipy import stats
def nap(pattern, md_loc=(100, 50)):
    rows = []
    for f in sorted(glob.glob(pattern)):
        for line in open(f):
            r = json.loads(line)
            if (r["M"], r["D"]) == md_loc: rows.append(r)
    return sorted(rows, key=lambda r: r["seed"])
for ten, pat in [("30 epoch (chính thức)", "rg3_all_s4*/all_detail.jsonl"),
                 ("300 epoch (kiểm tra)",  "rg4_e300_s4*/all_detail.jsonl")]:
    d = nap(pat)
    if not d: print(f"{ten}: chưa có dữ liệu"); continue
    g = np.concatenate([r["goc"] for r in d])
    print(f"\n=== {ten} — {len(d)} seed, gốc {g.mean():.3f} ===")
    for key, nhan in [("dauvao","TN1"), ("loc","TN2"), ("phanbo","TN3")]:
        dif = np.concatenate([np.array(r[key]) - np.array(r["goc"]) for r in d])
        per = [float(np.mean(np.array(r[key]) - np.array(r["goc"]))) for r in d]
        npos = sum(1 for x in per if x > 0)
        t, pt = stats.ttest_1samp(dif, 0); w, pw = stats.wilcoxon(dif[dif != 0])
        dat = "ĐẠT" if (pt < .05 and pw < .05 and npos == len(d)) else "không đạt"
        print(f"  {nhan}: Δ={dif.mean():+.3f} | p_t={pt:.1e} p_w={pw:.1e} | {npos}/{len(d)} seed dương -> {dat}")

In [ ]:
!zip -r -q ketqua_e300.zip rg4_e300_s4*
print("Xong — tải ketqua_e300.zip về để đối chiếu chi tiết.")

## Ghi chú
- Quy tắc khai trước: điền trung vị ngày; P/E ≤ 100; lọc ROEA < 10%; phân bổ w ∝ max(ROEA, 0);
  mã thiếu dữ liệu — giữ khi lọc, vốn 0 khi phân bổ. Không ranking, không Sortino/MDD — một thước Sharpe.
- Notebook này THAY THẾ ThucNghiem_Cuoi.ipynb (bản hai lần chạy, hai baseline).